In [1]:
import os
import json
import time
import random
from learning_helper import *
from preprocess import *

In [2]:
saved_anchor_dir = '/home/zhaomin/Documents/github/loc_tmp/saved_anchors/'
data_type = 'full'
annotation_type = 'gold'
mode = 'text_image'
vision_api = False

# load the instances and preprocess the instances
print('-' * 60 + '\n' + "Loading and preprocessing the tweets...")
instances = load_data(saved_anchor_dir, data_type, annotation_type='gold', add_annot_type=True)
additional_instances = load_data(saved_anchor_dir, data_type, annotation_type='silver', add_annot_type=True)
instances = preprocess_instances(instances, mode, vision_api)
additional_instances = preprocess_instances(additional_instances, mode, vision_api)
print("Done." + '\n' + '-' * 60)

# split the instances into train, development, and test
train_instances, dev_instances, test_instances = split_instances(instances, data_type, annotation_type='gold')
train_instances = train_instances + additional_instances
random.shuffle(train_instances)

------------------------------------------------------------
Loading and preprocessing the tweets...
Preprocessing the text and image via pretrained BERT-base model and VGG16 model...


/home/zhaomin/envs/torch/lib/python3.8/site-packages/PIL/Image.py:960: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Done.
Time spent for BERT&VGG16 preprocessing: 3m 33s
Preprocessing the text and image via pretrained BERT-base model and VGG16 model...
Done.
Time spent for BERT&VGG16 preprocessing: 1m 57s
Done.
------------------------------------------------------------


In [5]:
# get the label mapping
label_to_idx = {'Yes': 1, 'Other': 0}
idx_to_label = {1: 'Yes', 0: 'Other'}

# load the parameter setting file
para_filename = 'para_setting.json'
with open(para_filename, 'r') as para_file:
    para_dict = json.loads(para_file.read())
para = para_dict[mode]

# get the data loader for train, development, and test
batch_size = para['batch_size']
train_loader, dev_loader, test_loader = get_data_loader(train_instances, dev_instances, test_instances, batch_size)

# get the model (pretrained OR raw model)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# train and save the trained model
num_epochs = para['num_epochs']
patience = para['patience']

In [6]:
import torch.optim as optim
import torch.nn as nn

annot_type = annotation_type
model = get_model(mode, annot_type, vision_api, para).to(device)

# set the optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=1e-03)
criterion = nn.CrossEntropyLoss().to(device)

In [1]:
import torch.optim as optim
import torch.nn as nn
import torch
import json
from learning_helper import get_model

saved_anchor_dir = '/home/zhaomin/Documents/github/loc_tmp/saved_anchors/'
data_type = 'full'
annotation_type = 'gold+silver'
mode = 'text_image'
vision_api = False

annot_type = annotation_type
# load the parameter setting file
para_filename = 'para_setting.json'
with open(para_filename, 'r') as para_file:
    para_dict = json.loads(para_file.read())
para = para_dict[mode]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = get_model(mode, annot_type, vision_api, para).to(device)

# print(para)
print(model)

# # set the optimizer and loss function
# optimizer = optim.Adam(model.parameters(), lr=1e-03)
# criterion = nn.CrossEntropyLoss().to(device)

TextImageModel(
  (fc_bert): Linear(in_features=768, out_features=512, bias=True)
  (fc_vgg): Linear(in_features=1000, out_features=512, bias=True)
  (fc_both): Linear(in_features=1768, out_features=512, bias=True)
  (fc_last1): Linear(in_features=1538, out_features=512, bias=True)
  (fc_last2): Linear(in_features=512, out_features=2, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
)


In [27]:
%get_model()

UsageError: Line magic function `%get_model()` not found.


In [8]:
model

TextImageModel(
  (fc_bert): Linear(in_features=768, out_features=512, bias=True)
  (fc_vgg): Linear(in_features=1000, out_features=512, bias=True)
  (fc_both): Linear(in_features=1768, out_features=512, bias=True)
  (fc_last1): Linear(in_features=1538, out_features=512, bias=True)
  (fc_last2): Linear(in_features=512, out_features=2, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
)

In [9]:
def train(classifier, iterator, optimizer, criterion, device, label_to_idx):
    """
    train the model
    :param classifier: the model to be trained
    :param iterator: training data loader
    :param optimizer: optimizer
    :param criterion: loss function
    :param device: cpu or cuda
    :param label_to_idx: convert the label into index to use the CrossEntropy
    :return: the loss and accuracy of each epoch
    """

    epoch_loss = 0
    epoch_correct = 0
    num_instances = 0

    for batch_x, batch_y in iterator:

        # get the text representation and image representation
        feats = batch_x.squeeze(1).to(device)

        label_ids = torch.tensor([label_to_idx[label] for label in batch_y]).to(device)

        optimizer.zero_grad()

        # use the model to predict
        print(feats.shape)
        preds = classifier(feats)
        preds = preds.squeeze(1).to(device)

        loss = criterion(preds, label_ids)
        epoch_loss += loss.item()

        # back-propogate
        loss.backward()
        optimizer.step()

        # calculate the accuracy
        pred_ids = preds.argmax(dim=1)
        epoch_correct += pred_ids.eq(label_ids).sum().item()
        num_instances += len(batch_y)

    return epoch_loss / num_instances, epoch_correct / num_instances

In [10]:
best_valid_loss = float('inf')
check_stopping = 0
if vision_api:
    model_name = f'retrained_{mode}_{annot_type}_visionapi_classifier.pkl'
else:
    model_name = f'retrained_{mode}_{annot_type}_classifier.pkl'
for i in range(num_epochs):

    start_time = time.time()
    train_loss, train_acc = train(model, train_loader, optimizer, criterion, device, label_to_idx)
    dev_loss, dev_acc = evaluate(model, dev_loader, criterion, device, label_to_idx)
    end_time = time.time()

    elapsed_mins, elapsed_secs = epoch_time(start_time, end_time)

    print("-" * 60)
    print(f"Epoch: {i+1} || Epoch Time: {elapsed_mins}m {elapsed_secs}s")
    print(f"Epoch: {i+1} || Train loss: {train_loss:.02f}, Train Acc: {train_acc:.02f}")
    print(f"Epoch: {i+1} || Dev loss: {dev_loss:.02f}, Dev Acc: {dev_acc:.02f}")

    # check if we need to save the model
    if dev_loss < best_valid_loss:
        check_stopping = 0
        best_valid_loss = dev_loss
        torch.save(model, model_name)
    else:
        check_stopping += 1
        print(f"The loss on development set does not decrease")
        if check_stopping == patience:
            print("The loss on development set does not decrease, stop training!")
            break

torch.Size([128, 1770])


RuntimeError: mat1 dim 1 must match mat2 dim 0

In [13]:
import torch

a = torch.ones((1, 768+1000))
# b, c, d = torch.split(a, [1:768], dim=1)
b = a[:, :768]
c = a[:, 768:]
print(b.shape)
print(c.shape)
print(d.shape)

torch.Size([1, 768])
torch.Size([1, 1000])
torch.Size([1, 232])


In [2]:
saved_anchor_dir = '/home/zhaomin/Documents/saved_anchors'
data_type = 'full'
annotation_type = 'gold'

instances = load_data(saved_anchor_dir, data_type, annotation_type)

print("Getting the sentence representation via BERT...")
bert_start_time = time.time()
instances = add_bert_output(instances)
bert_end_time = time.time()
elapsed_mins, elapsed_secs = epoch_time(bert_start_time, bert_end_time)
print(f"Bert Time: {elapsed_mins}m {elapsed_secs}s")

print("Getting the image representation via VGG16...")
resnet_start_time = time.time()
# instances = add_vgg_output(instances)
resnet_end_time = time.time()
elapsed_mins, elapsed_secs = epoch_time(resnet_start_time, resnet_end_time)
print(f"VGG16 Time: {elapsed_mins}m {elapsed_secs}s")

train_instances, dev_instances, test_instances = split_instances(instances, data_type, annotation_type)

Getting the sentence representation via BERT...
Bert Time: 1m 13s
Getting the image representation via VGG16...
VGG16 Time: 0m 0s


In [4]:
instances[0]

{'Answer.Q1_annotator_0003': 'Yes',
 'Answer.Q2_annotator_0003': '4',
 'adjudicated_label': 'Yes',
 'HITId': '39XCQ6V3L4SNFZ0V49SDK8POVHE65X',
 'image_filename': 'springbreak2019_anchor_1106330449134174210.jpg',
 'json_filename': 'springbreak2019_anchor_1106330449134174210.json',
 'anchor_location': 'Houston',
 'anchor_id': 'springbreak2019_1106330449134174210_1106330449134174210',
 'event': 'springbreak2019',
 'anchor_url': 'http://www.cse.unt.edu/~blanco/screenshots/anchor/springbreak2019_1106330449134174210_anchor_1106330449134174210.png',
 'tweet_text': 'The Houston Track &amp; Field program opened the outdoor season with 10 titles Thursday at the Spring Break Invitational. #HTownSpeedCity\n\nFull recap: https://t.co/ojWmyJo8g5 https://t.co/rO1O789hZf',
 'image_filepath': '/home/zhaomin/Documents/saved_anchors/image_file/springbreak2019_anchor_1106330449134174210.jpg',
 'bert_output': tensor([[-3.7314e-01, -5.2052e-01,  2.0909e-01,  8.7853e-02, -2.1290e-01,
          -7.7381e-01,  

### build the model

In [5]:
import transformers as ppb
import torch.nn as nn
import torch
import torch.nn.functional as F
import torch.optim as optim
from model import TextImageModel, TextOnlyModel, ImageOnlyModel

In [6]:
labels = set([x['adjudicated_label'] for x in instances])
label_to_idx = {label: index for index, label in enumerate(labels)}
idx_to_label = {index: label for index, label in enumerate(labels)}

In [7]:
from torch.utils.data import Dataset, DataLoader

class TweetDataset(Dataset):
    # write a new class to load our dataset
    def __init__(self, instances):
        self.instances = instances

    def __getitem__(self, index):
        instance = self.instances[index]
        label = instance['adjudicated_label']
        # if instance has features from resnet and bert, then concatenate these two kinds of features
        if 'bert_output' in instance.keys() and 'vgg_output' in instance.keys():
            bert_output = instance['bert_output']
            resnet_output = instance['vgg_output']
            feat = torch.cat((bert_output, resnet_output), dim=1)
        if 'bert_output' in instance.keys() and 'vgg_output' not in instance.keys():
            feat = instance['bert_output']
        if 'bert_output' not in instance.keys() and 'vgg_output' in instance.keys():
            feat = instance['vgg_output']
        return feat, label

    def __len__(self):
        return len(self.instances)

In [8]:
train_dataset = TweetDataset(train_instances)
dev_dataset = TweetDataset(dev_instances)
test_dataset = TweetDataset(test_instances)
train_loader = DataLoader(train_dataset, batch_size=128)
dev_loader = DataLoader(dev_dataset, batch_size=128)
test_loader = DataLoader(test_dataset, batch_size=128)

In [9]:
vgg_feat_dim = 1000
bert_feat_dim = 768
output_dim = 2
hidden_dim = 512
dropout_rate = 0.2

In [10]:
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = torch.device('cpu')

fcn = TextImageModel(bert_feat_dim, vgg_feat_dim, output_dim, hidden_dim, dropout_rate).to(device)
optimizer = optim.Adam(fcn.parameters(), lr=1e-03)
criterion = nn.CrossEntropyLoss().to(device)

num_epoches = 500
early_stopping = 10
best_valid_loss = float('inf')
for i in range(num_epoches):
    
    start_time = time.time()
    train_loss, train_acc = train(fcn, train_loader, optimizer, criterion, device, label_to_idx)
    dev_loss, dev_acc = evaluate(fcn, dev_loader, criterion, device, label_to_idx)
    end_time = time.time()
    
    elapsed_mins, elapsed_secs = epoch_time(start_time, end_time)
    
    print("-" * 60)
    print(f"Epoch: {i+1} || Epoch Time: {elapsed_mins}m {elapsed_secs}s")
    print(f"Epoch: {i+1} || Train loss: {train_loss:.02f}, Train Acc: {train_acc:.02f}")
    print(f"Epoch: {i+1} || Dev loss: {dev_loss:.02f}, Dev Acc: {dev_acc:.02f}")
    
    # check if we need to save the model    
    if dev_loss < best_valid_loss:
        check_stopping = 0
        best_valid_loss = dev_loss
        torch.save(fcn, 'pretrained_models/pretrained_text_image_classifier.pkl')
    else:
        check_stopping += 1
        print(f"The loss on development set does not decrease")
        if check_stopping == early_stopping:
            print("The loss on development set does not decrease, stop training!")
            break
    
# test the train model
print("-" * 60)
print("The results on test set are as follows.")
test_loss, test_acc = evaluate(fcn, test_loader, criterion, device, label_to_idx)
print(f"Test loss: {test_loss:.02f}, Test Acc: {test_acc:.02f}")

------------------------------------------------------------
Epoch: 1 || Epoch Time: 0m 0s
Epoch: 1 || Train loss: 0.01, Train Acc: 0.57
Epoch: 1 || Dev loss: 0.01, Dev Acc: 0.61
------------------------------------------------------------
Epoch: 2 || Epoch Time: 0m 0s
Epoch: 2 || Train loss: 0.01, Train Acc: 0.60
Epoch: 2 || Dev loss: 0.01, Dev Acc: 0.61
------------------------------------------------------------
Epoch: 3 || Epoch Time: 0m 0s
Epoch: 3 || Train loss: 0.01, Train Acc: 0.62
Epoch: 3 || Dev loss: 0.01, Dev Acc: 0.60
The loss on development set does not decrease
------------------------------------------------------------
Epoch: 4 || Epoch Time: 0m 0s
Epoch: 4 || Train loss: 0.01, Train Acc: 0.63
Epoch: 4 || Dev loss: 0.01, Dev Acc: 0.60
The loss on development set does not decrease
------------------------------------------------------------
Epoch: 5 || Epoch Time: 0m 0s
Epoch: 5 || Train loss: 0.00, Train Acc: 0.64
Epoch: 5 || Dev loss: 0.01, Dev Acc: 0.59
The loss on d

In [11]:
from sklearn.metrics import classification_report

# make prediction and print the classification report
pred_labels = test_model(fcn, test_loader, idx_to_label, device)
gold_labels = [x['adjudicated_label'] for x in test_instances]
print('-' * 60)
print(classification_report(gold_labels, pred_labels))

# # save the predicted label
# prediction_filename = 'prediction.json'
# with open(prediction_filename, 'w') as pred_file:
#     for index, instance in enumerate(test_instances):
#         pred_label = pred_labels[index]
#         instance['pred_label'] = pred_label
#         if 'bert_output' in instance.keys():
#             del instance['bert_output']
#         if 'resnet_output' in instance.keys():
#             del instance['resnet_output']
#         pred_file.write(json.dumps(instance))
#         pred_file.write('\n')

------------------------------------------------------------
              precision    recall  f1-score   support

       Other       0.60      0.52      0.56       640
         Yes       0.59      0.67      0.63       668

    accuracy                           0.60      1308
   macro avg       0.60      0.60      0.59      1308
weighted avg       0.60      0.60      0.59      1308



**Try to load the saved model**

In [115]:
model_path = '/home/zhaomin/Documents/github/loc_tmp/pretrained_models/pretrained_text_only_classifier.pt'
test_device = torch.device('cpu')

# temp_fcn = TextImageModel(bert_feat_dim, output_dim, hidden_dim, dropout_rate)
# temp_fcn.load_state_dict(torch.load(model_path, map_location=test_device))
temp_fcn = torch.load(model_path).to(test_device)
# temp_fcn.eval()

from sklearn.metrics import classification_report

pred_labels = test_model(temp_fcn, test_loader, idx_to_label, test_device)
gold_labels = [x['adjudicated_label'] for x in test_instances]
print('-' * 60)
print(classification_report(gold_labels, pred_labels))

------------------------------------------------------------
              precision    recall  f1-score   support

       Other       0.64      0.61      0.63       640
         Yes       0.65      0.68      0.66       668

    accuracy                           0.65      1308
   macro avg       0.65      0.64      0.64      1308
weighted avg       0.65      0.65      0.64      1308



# test the scripts

In [1]:
import json
import os
import sys
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from learning_helper import majority_baseline, get_model, loop_epochs, test_model, save_prediction
from preprocess import load_data, preprocess_instances, split_instances, get_data_loader, check_argument
from sklearn.metrics import classification_report

In [2]:
pretrained_model_dir = 'pretrained_models'
annot_type = 'gold'
data_type = 'full'
mode = 'text_only'
model_type = 'pretrained'

torch.manual_seed(1234)
torch.cuda.manual_seed_all(1234)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

In [3]:
# load the instances
saved_anchor_dir = '/home/zhaomin/Documents/saved_anchors'
print('-' * 60 + '\n' + "Loading the tweets...")
if annot_type != 'gold+silver':
    instances = load_data(saved_anchor_dir, data_type, annot_type)
else:
    instances = load_data(saved_anchor_dir, data_type, annotation_type='gold')
    additional_instances = load_data(saved_anchor_dir, data_type, annotation_type='silver')
print("Done." + '\n' + '-' * 60)

# preprocess the instance
instances = preprocess_instances(instances, mode)
if annot_type == 'gold+silver':
    additional_instances = preprocess_instances(additional_instances, mode)

# split the instances into train, development, and test
if annot_type == 'gold+silver':
    train_instances, dev_instances, test_instances = split_instances(instances, data_type, annotation_type='gold')
    train_instances = train_instances + additional_instances
    print(len(train_instances))
else:
    train_instances, dev_instances, test_instances = split_instances(instances, data_type, annot_type)
    
# get the label set
instances = train_instances + dev_instances + test_instances
labels = set([x['adjudicated_label'] for x in instances])
label_to_idx = {label: index for index, label in enumerate(labels)}
idx_to_label = {index: label for index, label in enumerate(labels)}

# load the parameter setting file
para_filename = 'para_setting.json'
with open(para_filename, 'r') as para_file:
    para_dict = json.loads(para_file.read())
para = para_dict[mode]

# get the data loader for train, development, and test
batch_size = para['batch_size']
train_loader, dev_loader, test_loader = get_data_loader(train_instances, dev_instances, test_instances, batch_size)

------------------------------------------------------------
Loading the tweets...
Done.
------------------------------------------------------------
------------------------------------------------------------
Preprocessing the text via pretrained BERT-base model...
Done.
Time spent for BERT preprocessing: 1m 13s
------------------------------------------------------------


In [13]:
# get the model (pretrained OR raw model)
train_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
test_device = torch.device('cpu')

model_filename = f"pretrained_{mode}_classifier.pkl"
model_path = os.path.join(pretrained_model_dir, model_filename)
model = torch.load(model_path)

# make prediction and print the classification report
model = model.to(test_device)
model.eval()
pred_labels = test_model(model, test_loader, idx_to_label, test_device)
gold_labels = [x['adjudicated_label'] for x in test_instances]
print('-' * 60)
print(classification_report(gold_labels, pred_labels))

------------------------------------------------------------
              precision    recall  f1-score   support

       Other       0.64      0.61      0.63       640
         Yes       0.64      0.67      0.66       668

    accuracy                           0.64      1308
   macro avg       0.64      0.64      0.64      1308
weighted avg       0.64      0.64      0.64      1308



In [10]:
tag_dir = 'saved_anchors/tag_file'
tags = []
filenames = os.listdir(tag_dir)
for filename in filenames:
    filepath = os.path.join(tag_dir, filename)
    with open(filepath, 'r') as tagfile:
        tag_list = json.load(tagfile)
    tag = [x['tag'] for x in tag_list]
    tags.append(tag)
tags_count = [len(x) for x in tags]

In [15]:
import statistics

print(f"Mean: {statistics.mean(tags_count)}")
print(f"Median: {statistics.median(tags_count)}")
print(f"Mode: {statistics.mode(tags_count)}")

Mean: 9.337096454743513
Median: 10.0
Mode: 10


In [17]:
max(tags_count)

10